# GeoCrackVision V3 — MobileNetV2-UNet

Train a Rock crack segmenter Model from scratch with a **MobileNetV2** encoder in place of ResNet34.

| | V2 (current) | V3 (this notebook) |
|---|---|---|
| Encoder | resnet34 (ImageNet) | mobilenet_v2 (ImageNet) |
| Params | 24.4 M | ~6.6 M |
| ONNX | 98 MB | ~26 MB |
| CPU / 224 tile (dev box, ORT) | 48 ms | measured below |

**Held constant so the encoder is the only variable:** dataset + splits, augmentation, `GeoCrackLoss`
(0.25 boundary-BCE + 0.74 Dice + 0.01 continuity), AdamW + ReduceLROnPlateau, fp16 AMP, early stopping.

**Deployment contract is unchanged:** input `image [B,3,224,224]` float32, ImageNet mean/std, no CLAHE,
output raw `logits [B,1,224,224]`, threshold 0.3. The backend only needs `MODEL_PATH` / `MODEL_CONFIG_PATH` swapped.

**Fixes vs V2**
- Metrics are reported at the **deploy threshold 0.3** as well as 0.5, both pooled over all pixels
  (dataset-level) and per-image averaged (V2's formula, for comparability with 0.6086).
- Best checkpoint is chosen by val IoU **at 0.3**, the threshold the app actually uses.
- Full resume state (model, optimizer, scheduler, GradScaler, history, early stopping) saved to Drive every epoch —
  a Colab disconnect costs at most one epoch. Just re-run all cells.
- ONNX exported with opset 17, legacy exporter (single file), verified against PyTorch and benchmarked on CPU.
- Side-by-side test evaluation against the V2 ResNet34 checkpoint on the same split.

In [ ]:
!pip install -q segmentation-models-pytorch onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 20.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import shutil
import zipfile

from pathlib import Path

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

In [ ]:
# Check TORCH VERSION
import segmentation_models_pytorch as smp

print(f'torch {torch.__version__} | smp {smp.__version__}')

torch 2.11.0+cu130 | smp 0.5.0


## Paths (Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path from Drive to Colab

DRIVE_ROOT  = Path('/content/drive/MyDrive/geocrackai')
ZIP_PATH    = DRIVE_ROOT / 'patched_images.zip'
CSV_TRAIN   = DRIVE_ROOT / 'train_final.csv'
CSV_VAL     = DRIVE_ROOT / 'validation_final.csv'
CSV_TEST    = DRIVE_ROOT / 'test_final.csv'

# V2 ResNet34 checkpoint, used only for the side-by-side test comparison
BASELINE_CKPT = DRIVE_ROOT / 'model_output8' / 'checkpoints' / 'best_iou.pt'

DRIVE_OUT   = DRIVE_ROOT / 'model_output_v3_mnv2'
CHECKPOINTS = DRIVE_OUT / 'checkpoints'
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT  = Path('/content/geocrack')
IMAGES_DIR  = LOCAL_ROOT / 'patched_images'

for p in [ZIP_PATH, CSV_TRAIN, CSV_VAL, CSV_TEST, BASELINE_CKPT]:
    print(f'{"OK " if p.exists() else "MISSING"}  {p}')
print(f'Outputs -> {DRIVE_OUT}')

OK   /content/drive/MyDrive/geocrackai/patched_images.zip
OK   /content/drive/MyDrive/geocrackai/train_final.csv
OK   /content/drive/MyDrive/geocrackai/validation_final.csv
OK   /content/drive/MyDrive/geocrackai/test_final.csv
OK   /content/drive/MyDrive/geocrackai/model_output8/checkpoints/best_iou.pt
Outputs -> /content/drive/MyDrive/geocrackai/model_output_v3_mnv2


In [ ]:
# Unzip to local disk: reading 25k small PNGs from Drive FUSE would starve the GPU
if not IMAGES_DIR.exists():
    print('Unzip patched_images.zip ...')
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_ROOT)
    print(f'Listo en {time.time() - t0:.1f}s')
else:
    print(f'Directorio ya existe ({IMAGES_DIR}), saltando descompresion.')

n_orig = len(list(IMAGES_DIR.glob('*original_patch*.png')))
n_mask = len(list(IMAGES_DIR.glob('*binarymask_patch*.png')))
print(f'Original Images : {n_orig:,}')
print(f'Binary Masks    : {n_mask:,}')

Unzip patched_images.zip ...
Listo en 23.4s
Original Images : 12,493
Binary Masks    : 12,493


## Config

In [ ]:
CFG = {
    # Model
    'arch':            'Unet',
    'encoder':         'mobilenet_v2',
    'encoder_weights': 'imagenet',
    'in_channels':     3,
    'num_classes':     1,
    'img_size':        224,

    # Hyperparams
    'epochs':          120,
    'batch_size':      64,
    'num_workers':     min(4, os.cpu_count() or 2),
    'lr_initial':      5e-4,
    'lr_min':          1e-5,
    'weight_decay':    1e-4,
    'lr_factor':       0.5,
    'lr_patience':     6,
    'lr_threshold':    2e-4,
    'grad_clip':       1.0,
    'amp_dtype':       torch.float16,   # global config

    # Loss weights (identical to V2)
    'w_bce_boundary':  0.25,
    'w_dice':          0.74,
    'w_continuity':    0.01,
    'boundary_kernel': 3,
    'boundary_weight': 5.0,

    # Early stopping (on val loss)
    'patience':        20,
    'min_delta':       1e-4,

    # Thresholds: the app thresholds at 0.3; 0.5
    'deploy_threshold': 0.3,
    'eval_thresholds':  (0.3, 0.5),

    # Normalizacion ImageNet
    'mean':            [0.485, 0.456, 0.406],
    'std':             [0.229, 0.224, 0.225],

    'seed':            42,
}

SELECT_KEY = f"iou@{CFG['deploy_threshold']:.1f}"   # metric used to pick the best checkpoint

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('CUDA no disponible — entrenando en CPU (sin AMP). Runtime > Change runtime type > T4 GPU.')
    CFG['amp_dtype'] = None

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


## Dataset (same as V2)

In [ ]:
class GeoCrackDataset(Dataset):
    '''
    Pairs (original patch + binary mask) from CSV columns
    image_path : *_original_patch*.png
    mask_path  : *_binarymask_patch*.png
    Only the file name is used; files are resolved inside images_dir.
    '''
    def __init__(self, csv_path,
                 images_dir,
                 transform=None,
                 augment=False):

        self.df         = pd.read_csv(csv_path)
        self.images_dir = images_dir
        self.transform  = transform
        self.augment    = augment

        self.df['_img']  = self.df['image_path'].apply(lambda p: Path(p).name)
        self.df['_mask'] = self.df['mask_path'].apply(lambda p: Path(p).name)

        missing = sum(
            (not (images_dir / a).exists()) + (not (images_dir / b).exists())
            for a, b in zip(self.df['_img'], self.df['_mask'])
        )
        if missing:
            print(f'Warning: {missing} files not found in {images_dir}')

    def __len__(self):
        return len(self.df)

    def _augment(self, image, mask):
        if torch.rand(1) > 0.5:
            image, mask = TF.hflip(image), TF.hflip(mask)
        if torch.rand(1) > 0.5:
            image, mask = TF.vflip(image), TF.vflip(mask)

        # Rotation 0 / 90 / 180 / 270
        angle = torch.randint(0, 4, (1,)).item() * 90
        if angle:
            image, mask = TF.rotate(image, angle), TF.rotate(mask, angle)

        if torch.rand(1) > 0.5:

            image = TF.adjust_brightness(image, 0.85 + torch.rand(1).item() * 0.3)

        # NOTE: kept identical to V2 (0.65-0.90 only ever lowers contrast) so the
        # comparison isolates the encoder. Widen to e.g. 0.75-1.25 in a later run.
        if torch.rand(1) > 0.5:
            image = TF.adjust_contrast(image, 0.65 + torch.rand(1).item() * 0.25)

        return image, mask

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(self.images_dir / row['_img']).convert('RGB')
        mask  = Image.open(self.images_dir / row['_mask']).convert('L')

        if self.augment:
            image, mask = self._augment(image, mask)

        image = self.transform(image) if self.transform else TF.to_tensor(image)
        mask  = (TF.to_tensor(mask) > 0.5).float()   # [1, 224, 224] binary

        return image, mask

In [ ]:
img_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=CFG['mean'], std=CFG['std']),
])

train_dataset = GeoCrackDataset(CSV_TRAIN, IMAGES_DIR, img_transform, augment=True)
val_dataset   = GeoCrackDataset(CSV_VAL,   IMAGES_DIR, img_transform, augment=False)
test_dataset  = GeoCrackDataset(CSV_TEST,  IMAGES_DIR, img_transform, augment=False)

loader_kw = dict(
    batch_size  = CFG['batch_size'],
    num_workers = CFG['num_workers'],
    pin_memory  = DEVICE.type == 'cuda',
    persistent_workers = CFG['num_workers'] > 0,
)
train_loader = DataLoader(train_dataset, shuffle=True,  drop_last=True, **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)

print(f'train {len(train_dataset):,} | val {len(val_dataset):,} | test {len(test_dataset):,}')

train 9,726 | val 1,823 | test 609


## Loss (same as V2)

In [ ]:
class BoundaryWeightedBCE(nn.Module):

    def __init__(self, kernel_size=3, boundary_weight=5.0):

        super().__init__()
        self.kernel_size     = kernel_size
        self.boundary_weight = boundary_weight
        self.register_buffer('dilation_kernel', torch.ones(1, 1, kernel_size, kernel_size))

    def _get_boundary_map(self, mask):
        pad     = self.kernel_size // 2
        k       = self.dilation_kernel.to(mask.dtype)
        dilated = (F.conv2d(mask, k, padding=pad) > 0).float()
        eroded  = 1.0 - F.conv2d(1.0 - mask, k, padding=pad).clamp(0, 1)
        eroded  = (eroded > 0.5).float()
        boundary = (dilated - eroded).clamp(0, 1)

        return 1.0 + boundary * self.boundary_weight

    def forward(self, logits, targets):
        with torch.no_grad():
            weight_map = self._get_boundary_map(targets)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        return (bce * weight_map).mean()


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs        = torch.sigmoid(logits).view(logits.size(0), -1)
        targets_flat = targets.view(targets.size(0), -1)
        intersection = (probs * targets_flat).sum(dim=1)
        union        = probs.sum(dim=1) + targets_flat.sum(dim=1)
        dice_score   = (2.0 * intersection + self.smooth) / (union + self.smooth)

        return 1.0 - dice_score.mean()


In [ ]:
class ContinuityLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('sobel_x', torch.tensor(
            [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
            dtype=torch.float32).view(1, 1, 3, 3))
        self.register_buffer('sobel_y', torch.tensor(
            [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]
            , dtype=torch.float32).view(1, 1, 3, 3))
        self.register_buffer('laplacian', torch.tensor(
            [[0, 1, 0], [1, -4, 1], [0, 1, 0]],
            dtype=torch.float32).view(1, 1, 3, 3))

    def _gradient_magnitude(self, x):
        gx = F.conv2d(x, self.sobel_x.to(x.dtype), padding=1)
        gy = F.conv2d(x, self.sobel_y.to(x.dtype), padding=1)

        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-6)

    def _laplacian_magnitude(self, x):
        return F.conv2d(x, self.laplacian.to(x.dtype), padding=1).abs()

    def forward(self, logits, targets):
        # Siempre en float32 — evita overflow en float16
        probs   = torch.sigmoid(logits).float()
        targets = targets.float()

        grad_loss = F.mse_loss(self._gradient_magnitude(probs), self._gradient_magnitude(targets))

        fracture_zone = (targets > 0.5).float()
        zone_sum      = fracture_zone.sum()
        if zone_sum < 1.0:
            lap_loss = torch.tensor(0.0, device=logits.device)
        else:
            lap_loss = F.mse_loss(
                self._laplacian_magnitude(probs) * fracture_zone,
                self._laplacian_magnitude(targets) * fracture_zone,
                reduction='sum',
            ) / zone_sum.clamp(min=1.0)

        total = grad_loss + 0.5 * lap_loss
        if not torch.isfinite(total):
            return torch.tensor(0.0, device=logits.device, requires_grad=True )

        return total



## Metrics

Two views, at each threshold:
- `iou@t`, `f1@t`, `precision@t`, `recall@t` — **pooled over every pixel in the split** (dataset-level).
  Not distorted by crack-free patches.
- `img_iou@t`, `img_f1@t` — per-image score averaged over images, exactly V2's `compute_metrics`.
  A crack-free patch with an empty prediction scores 1.0 here, so it moves with the share of empty patches.
  V2's reported 0.6086 is `img_iou@0.5`.

In [ ]:
class SegMetrics:
    def __init__(self, thresholds):
        self.thresholds = tuple(thresholds)
        self.reset()

    def reset(self):
        self.counts   = {t: torch.zeros(4, dtype=torch.float64) for t in self.thresholds}  # tp fp fn tn
        self.img_iou  = {t: 0.0 for t in self.thresholds}
        self.img_f1   = {t: 0.0 for t in self.thresholds}
        self.n_images = 0

    @torch.no_grad()
    def update(self, logits, targets):
        probs  = torch.sigmoid(logits.float())
        t_flat = targets.float().view(targets.size(0), -1)
        for t in self.thresholds:
            p_flat = (probs > t).float().view(probs.size(0), -1)
            tp = (p_flat * t_flat).sum(1)
            fp = (p_flat * (1 - t_flat)).sum(1)
            fn = ((1 - p_flat) * t_flat).sum(1)
            tn = ((1 - p_flat) * (1 - t_flat)).sum(1)
            self.counts[t] += torch.stack([tp.sum(),
                                           fp.sum(),
                                           fn.sum(),
                                           tn.sum()]
                                          ).double().cpu()
            self.img_iou[t] += ((tp + 1e-8) / (tp + fp + fn + 1e-8)).sum().item()
            self.img_f1[t]  += ((2 * tp + 1e-8) / (2 * tp + fp + fn + 1e-8)).sum().item()
        self.n_images += targets.size(0)

    def compute(self):
        out = {}
        n = max(self.n_images, 1)
        for t in self.thresholds:
            tp, fp, fn, tn = self.counts[t].tolist()
            k = f'{t:.1f}'
            out[f'iou@{k}']       = tp / max(tp + fp + fn, 1.0)
            out[f'f1@{k}']        = 2 * tp / max(2 * tp + fp + fn, 1.0)
            out[f'precision@{k}'] = tp / max(tp + fp, 1.0)
            out[f'recall@{k}']    = tp / max(tp + fn, 1.0)
            out[f'px_acc@{k}']    = (tp + tn) / max(tp + fp + fn + tn, 1.0)
            out[f'img_iou@{k}']   = self.img_iou[t] / n
            out[f'img_f1@{k}']    = self.img_f1[t] / n
        return out

## Model

In [ ]:
def build_model(encoder, encoder_weights):
    return smp.Unet(
        encoder_name    = encoder,
        encoder_weights = encoder_weights,
        in_channels     = CFG['in_channels'],
        classes         = CFG['num_classes'],
        activation      = None,  # raw logits — sigmoid in loss / at inference
    )


In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())


model = build_model(CFG['encoder'], CFG['encoder_weights']).to(DEVICE)
n_enc = count_params(model.encoder)
n_all = count_params(model)
print(f"Encoder  {CFG['encoder']}: {n_enc / 1e6:.2f} M params")
print(f'Total UNet         : {n_all / 1e6:.2f} M params  (V2 ResNet34-UNet: 24.44 M)')
print(f'fp32 weights       : {n_all * 4 / 1e6:.1f} MB')

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Encoder  mobilenet_v2: 2.22 M params
Total UNet         : 6.63 M params  (V2 ResNet34-UNet: 24.44 M)
fp32 weights       : 26.5 MB


In [ ]:
criterion = GeoCrackLoss(CFG).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG['lr_initial'], weight_decay=CFG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=CFG['lr_factor'], patience=CFG['lr_patience'],
    threshold=CFG['lr_threshold'], min_lr=CFG['lr_min'],
)
AMP_ENABLED = DEVICE.type == 'cuda' and CFG['amp_dtype'] is not None
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED, init_scale=2 ** 14, growth_interval=200)


In [ ]:

print(f"Optimizer : AdamW lr={CFG['lr_initial']:.0e} wd={CFG['weight_decay']:.0e}")
print(f"Scheduler : ReduceLROnPlateau factor={CFG['lr_factor']} patience={CFG['lr_patience']} min_lr={CFG['lr_min']:.0e}")
print(f"Loss      : {CFG['w_bce_boundary']}*BCE_boundary + {CFG['w_dice']}*Dice + {CFG['w_continuity']}*Continuity")
print(f"AMP       : {CFG['amp_dtype'] if AMP_ENABLED else 'off'}")

Optimizer : AdamW lr=5e-04 wd=1e-04
Scheduler : ReduceLROnPlateau factor=0.5 patience=6 min_lr=1e-05
Loss      : 0.25*BCE_boundary + 0.74*Dice + 0.01*Continuity
AMP       : torch.float16


## Early stopping + checkpointing (resumable)

In [ ]:
class EarlyStopping:
    '''Stops on val loss; tracks best val IoU (at the deploy threshold) separately.'''

    def __init__(self, patience, min_delta):
        self.patience  = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.best_iou  = -1.0
        self.best_epoch = 0
        self.counter   = 0
        self.stop      = False

    def step(self, val_loss, val_iou, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f'  [ >>] No improvement in val loss {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.stop = True
                print('  EARLY STOPPING')

        iou_improved = val_iou > self.best_iou
        if iou_improved:
            self.best_iou   = val_iou
            self.best_epoch = epoch
        return iou_improved

    def state_dict(self):
        return dict(self.__dict__)

    def load_state_dict(self, state):
        self.__dict__.update(state)



In [ ]:

early_stopping = EarlyStopping(CFG['patience'], CFG['min_delta'])
history = {}

LAST_CKPT    = CHECKPOINTS / 'last.pt'
BEST_CKPT    = CHECKPOINTS / 'best_iou.pt'
HISTORY_JSON = CHECKPOINTS / 'training_history.json'

CFG_SERIAL = {k: (str(v) if isinstance(v, torch.dtype) else v) for k, v in CFG.items()}



In [ ]:
## Training Helpers
def _atomic_save(obj, path):
    tmp = path.with_suffix('.tmp')
    torch.save(obj, tmp)
    tmp.replace(path)


def save_last(epoch):
    _atomic_save({
        'epoch':           epoch,
        'cfg':             CFG_SERIAL,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state':    scaler.state_dict(),
        'early_stopping':  early_stopping.state_dict(),
        'history':         history,
    }, LAST_CKPT)


def save_best(epoch, val_metrics):
    _atomic_save({
        'epoch':       epoch,
        'cfg':         CFG_SERIAL,
        'model_state': model.state_dict(),
        'val_metrics': val_metrics,
    }, BEST_CKPT)


In [ ]:
# SSecurity checkpoints before commit.
START_EPOCH = 1
if LAST_CKPT.exists():
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE, weights_only=False)
    if ckpt['cfg']['encoder'] != CFG['encoder']:
        raise RuntimeError(f"{LAST_CKPT} is for encoder {ckpt['cfg']['encoder']}, not {CFG['encoder']}")
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    early_stopping.load_state_dict(ckpt['early_stopping'])
    history     = ckpt['history']
    START_EPOCH = ckpt['epoch'] + 1
    print(f'Resuming from epoch {START_EPOCH} (best {SELECT_KEY}={early_stopping.best_iou:.4f} @ epoch {early_stopping.best_epoch})')
else:
    print('No checkpoint found — training from scratch (ImageNet encoder).')

# Drive write check
(CHECKPOINTS / 'write_test.txt').write_text('ok')
(CHECKPOINTS / 'write_test.txt').unlink()
print('Escritura en Drive verificada.')

No checkpoint found — training from scratch (ImageNet encoder).
Escritura en Drive verificada.


## Train / validate

In [ ]:
def train_one_epoch(model, loader,
                    criterion, optimizer,
                    scaler, device,
                    amp_dtype):
    model.train()
    metrics = SegMetrics((CFG['deploy_threshold'],))
    total_loss, n_batches, n_skipped = 0.0, 0, 0
    comps_sum = {'loss_bce_boundary': 0.0, 'loss_dice': 0.0, 'loss_continuity': 0.0}

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=AMP_ENABLED):
            logits      = model(images)
            loss, comps = criterion(logits, masks)

        if not torch.isfinite(loss):
            n_skipped += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()

        metrics.update(logits.detach(), masks)
        total_loss += loss.item()
        for k in comps_sum:
            comps_sum[k] += comps[k]
        n_batches += 1

    if n_skipped:
        print(f'  ADVERTENCIA: {n_skipped} batches con loss no finito saltados')
    n = max(n_batches, 1)
    out = {'loss': total_loss / n, **{k: v / n for k, v in comps_sum.items()}}
    out.update(metrics.compute())
    out['scaler_scale'] = scaler.get_scale()

    return out


In [ ]:
# Evaluate Metrics
@torch.no_grad()
def evaluate(model, loader,
             criterion,
             device,
             thresholds
             , amp_dtype=None):

    model.eval()
    metrics = SegMetrics(thresholds)

    total_loss, n_batches = 0.0, 0
    amp = device.type == 'cuda' and amp_dtype is not None

    for images, masks in loader:

        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp):

            logits  = model(images)
            loss, _ = criterion(logits, masks)

        metrics.update(logits, masks)
        total_loss += loss.item()
        n_batches  += 1

    return {'loss': total_loss / max(n_batches, 1), **metrics.compute()}

In [ ]:
def log_history(prefix, metrics):
    for k, v in metrics.items():
        history.setdefault(f'{prefix}_{k}', []).append(v)



In [ ]:

print('=' * 70)
print(f"TRAINING {CFG['encoder']}-UNet  epochs {START_EPOCH}..{CFG['epochs']}  select by val {SELECT_KEY}")
print('=' * 70)

t_start = time.time()
for epoch in range(START_EPOCH, CFG['epochs'] + 1):
    if early_stopping.stop:
        print('Early stopping already triggered in a previous session.')
        break
    t_epoch = time.time()

    tr = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, CFG['amp_dtype'])
    va = evaluate(model, val_loader, criterion, DEVICE, CFG['eval_thresholds'], CFG['amp_dtype'])
    scheduler.step(va['loss'])

    lr = optimizer.param_groups[0]['lr']
    scale = tr.pop('scaler_scale')
    log_history('train', tr)
    log_history('val', va)
    history.setdefault('lr', []).append(lr)
    history.setdefault('epoch', []).append(epoch)

    k = f"{CFG['deploy_threshold']:.1f}"
    print(f"\nEpoch {epoch:03d}/{CFG['epochs']} | {time.time() - t_epoch:.0f}s | {(time.time() - t_start) / 60:.1f}min")
    print(f"  Train loss={tr['loss']:.4f}  IoU@{k}={tr[f'iou@{k}']:.4f}  F1@{k}={tr[f'f1@{k}']:.4f}")
    print(f"  Val   loss={va['loss']:.4f}  IoU@{k}={va[f'iou@{k}']:.4f}  F1@{k}={va[f'f1@{k}']:.4f}  "
          f"P={va[f'precision@{k}']:.3f} R={va[f'recall@{k}']:.3f}  imgIoU@0.5={va['img_iou@0.5']:.4f}")
    print(f"  LR={lr:.2e}  scale={scale:.0f}  |  BCE={tr['loss_bce_boundary']:.4f}  "
          f"Dice={tr['loss_dice']:.4f}  Cont={tr['loss_continuity']:.4f}")

    if early_stopping.step(va['loss'], va[SELECT_KEY], epoch):
        save_best(epoch, va)
        print(f'  Best model saved ({SELECT_KEY}={va[SELECT_KEY]:.4f})')

    save_last(epoch)
    HISTORY_JSON.write_text(json.dumps(history))

    if early_stopping.stop:
        print(f'\nEntrenamiento detenido en epoch {epoch} por early stopping.')
        break

print('\n' + '=' * 70)
print(f'Best val {SELECT_KEY}: {early_stopping.best_iou:.4f} @ epoch {early_stopping.best_epoch}')
print(f'Best val loss  : {early_stopping.best_loss:.4f}')
print('=' * 70)

TRAINING mobilenet_v2-UNet  epochs 1..120  select by val iou@0.3

Epoch 001/120 | 75s | 1.2min
  Train loss=0.5327  IoU@0.3=0.2279  F1@0.3=0.3711
  Val   loss=0.3534  IoU@0.3=0.5497  F1@0.3=0.7094  P=0.610 R=0.848  imgIoU@0.5=0.5411
  LR=5.00e-04  scale=16384  |  BCE=0.3730  Dice=0.5832  Cont=0.7857
  Best model saved (iou@0.3=0.5497)

Epoch 002/120 | 65s | 2.3min
  Train loss=0.3303  IoU@0.3=0.5739  F1@0.3=0.7293
  Val   loss=0.3114  IoU@0.3=0.6008  F1@0.3=0.7507  P=0.701 R=0.808  imgIoU@0.5=0.5743
  LR=5.00e-04  scale=32768  |  BCE=0.2833  Dice=0.3416  Cont=0.6673
  Best model saved (iou@0.3=0.6008)

Epoch 003/120 | 64s | 3.4min
  Train loss=0.3144  IoU@0.3=0.5856  F1@0.3=0.7387
  Val   loss=0.3026  IoU@0.3=0.5937  F1@0.3=0.7450  P=0.665 R=0.847  imgIoU@0.5=0.5800
  LR=5.00e-04  scale=65536  |  BCE=0.2815  Dice=0.3210  Cont=0.6506

Epoch 004/120 | 63s | 4.5min
  Train loss=0.3038  IoU@0.3=0.5975  F1@0.3=0.7480
  Val   loss=0.2957  IoU@0.3=0.5946  F1@0.3=0.7458  P=0.657 R=0.862  imgIo

In [ ]:
## Plot Training Curves

## Test set: V3 vs V2 (same split, same metrics)

In [ ]:
k = f"{CFG['deploy_threshold']:.1f}"
ep = history['epoch']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"GeoCrack V3 {CFG['encoder']}-UNet — training curves", fontsize=14)

axes[0, 0].plot(ep, history['train_loss'], label='Train')
axes[0, 0].plot(ep, history['val_loss'],   label='Val')
axes[0, 0].set_title('Loss total')

axes[0, 1].plot(ep, history[f'train_iou@{k}'], label='Train')
axes[0, 1].plot(ep, history[f'val_iou@{k}'],   label='Val')
axes[0, 1].plot(ep, history['val_img_iou@0.5'], label='Val per-image @0.5 (V2 metric)', ls='--')
axes[0, 1].set_title(f'IoU @ {k}')

axes[1, 0].plot(ep, history[f'val_precision@{k}'], label='Precision')
axes[1, 0].plot(ep, history[f'val_recall@{k}'],    label='Recall')
axes[1, 0].set_title(f'Val precision / recall @ {k}')

axes[1, 1].plot(ep, history['lr'], color='orange')
axes[1, 1].set_title('Learning Rate')
axes[1, 1].set_yscale('log')

for ax in axes.flat:
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend()

plt.tight_layout()
plt.savefig(str(CHECKPOINTS / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(best['model_state'])
print(f"V3 best checkpoint: epoch {best['epoch']}, val {SELECT_KEY}={best['val_metrics'][SELECT_KEY]:.4f}")

results = {'V3 ' + CFG['encoder']: evaluate(model, test_loader, criterion, DEVICE, CFG['eval_thresholds'])}
params  = {'V3 ' + CFG['encoder']: count_params(model)}

baseline = None
if BASELINE_CKPT.exists():
    baseline = build_model('resnet34', None).to(DEVICE)
    b_ckpt = torch.load(BASELINE_CKPT, map_location=DEVICE, weights_only=False)
    baseline.load_state_dict(b_ckpt['model_state'])
    results['V2 resnet34'] = evaluate(baseline, test_loader, criterion, DEVICE, CFG['eval_thresholds'])
    params['V2 resnet34']  = count_params(baseline)
    print(f"V2 baseline: epoch {b_ckpt['epoch']}")
else:
    print(f'Baseline checkpoint not found ({BASELINE_CKPT}) — V3 only.')

rows = ['loss'] + [f'{m}@{t:.1f}' for t in CFG['eval_thresholds']
                   for m in ('iou', 'f1', 'precision', 'recall', 'img_iou', 'img_f1')]
table = pd.DataFrame({name: [r[k] for k in rows] for name, r in results.items()}, index=rows)
table.loc['params (M)'] = [params[n] / 1e6 for n in table.columns]
print('\nRESULTADOS — TEST SET')
print(table.round(4).to_string())
print('\nimg_iou@0.5 is the metric V2 reported as 0.6086.')

V3 best checkpoint: epoch 49, val iou@0.3=0.6348
V2 baseline: epoch 94

RESULTADOS — TEST SET
               V3 mobilenet_v2  V2 resnet34
loss                    0.2761       0.2730
iou@0.3                 0.6279       0.6288
f1@0.3                  0.7714       0.7721
precision@0.3           0.7221       0.7196
recall@0.3              0.8279       0.8329
img_iou@0.3             0.6058       0.6066
img_f1@0.3              0.7404       0.7418
iou@0.5                 0.6326       0.6330
f1@0.5                  0.7749       0.7753
precision@0.5           0.7559       0.7491
recall@0.5              0.7949       0.8033
img_iou@0.5             0.6079       0.6094
img_f1@0.5              0.7419       0.7441
params (M)              6.6289      24.4364

img_iou@0.5 is the metric V2 reported as 0.6086.


In [ ]:
# Visual check on test patches that actually contain cracks
k = CFG['deploy_threshold']
crack_idx = [i for i in range(len(test_dataset)) if test_dataset[i][1].sum() > 50][:6]
mean = torch.tensor(CFG['mean']).view(3, 1, 1)
std  = torch.tensor(CFG['std']).view(3, 1, 1)


In [ ]:
# Visual Image check : crack by crack
cols = 4 if baseline is not None else 3
fig, axes = plt.subplots(len(crack_idx), cols, figsize=(3.2 * cols, 3.2 * len(crack_idx)))
model.eval()
with torch.no_grad():
    for r, i in enumerate(crack_idx):
        img, gt = test_dataset[i]
        x = img.unsqueeze(0).to(DEVICE)
        panels = [
            ((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy(), 'image'),
            (gt[0].numpy(), 'ground truth'),
            ((torch.sigmoid(model(x))[0, 0] > k).cpu().numpy(), f'V3 @ {k}'),
        ]
        if baseline is not None:
            panels.append(((torch.sigmoid(baseline(x))[0, 0] > k).cpu().numpy(), f'V2 @ {k}'))
        for c, (arr, title) in enumerate(panels):
            axes[r, c].imshow(arr, cmap=None if arr.ndim == 3 else 'gray')
            axes[r, c].set_title(title)
            axes[r, c].axis('off')
plt.tight_layout()
plt.savefig(str(CHECKPOINTS / 'test_samples.png'), dpi=120, bbox_inches='tight')
plt.show()

## Export ONNX (deployment contract)

Legacy TorchScript exporter (`dynamo=False`): the dynamo exporter in V2 wrote a 0.3 MB file with the weights
left out. Opset 17, input `image`, output `logits`, dynamic batch axis — identical I/O to `geo_vision.onnx`.

In [ ]:
import onnx
import onnxruntime as ort

ONNX_NAME = f"geo_vision_{CFG['encoder']}.onnx"
onnx_path = DRIVE_OUT / ONNX_NAME

In [ ]:
# Explort Model ONNX runtime format
model.eval().cpu()
dummy = torch.randn(1, 3, CFG['img_size'], CFG['img_size'])
torch.onnx.export(
    model, dummy, str(onnx_path),
    export_params       = True,
    opset_version       = 17,
    do_constant_folding = True,
    input_names         = ['image'],
    output_names        = ['logits'],
    dynamic_axes        = {'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    dynamo              = False,
)


In [ ]:

size_mb = onnx_path.stat().st_size / 1e6
print(f'{onnx_path.name}: {size_mb:.1f} MB')
assert size_mb > 0.8 * count_params(model) * 4 / 1e6, 'ONNX too small — weights were not embedded'

onnx.checker.check_model(onnx.load(str(onnx_path)))
print('onnx.checker: OK')

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
for b in (1, 4):
    x = np.random.randn(b, 3, CFG['img_size'], CFG['img_size']).astype(np.float32)
    ort_out = sess.run(None, {'image': x})[0]
    with torch.no_grad():
        torch_out = model(torch.from_numpy(x)).numpy()
    diff = np.abs(torch_out - ort_out).max()
    print(f'batch={b}: output {ort_out.shape}, max |PyTorch - ONNX| = {diff:.2e}')
    assert ort_out.shape == (b, 1, CFG['img_size'], CFG['img_size'])
    assert diff < 1e-3, 'PyTorch / ONNX mismatch'
print('Consistencia PyTorch-ONNX: OK')

/tmp/ipykernel_1191/4093697689.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


geo_vision_mobilenet_v2.onnx: 26.5 MB
onnx.checker: OK
batch=1: output (1, 1, 224, 224), max |PyTorch - ONNX| = 1.34e-05
batch=4: output (4, 1, 224, 224), max |PyTorch - ONNX| = 1.81e-05
Consistencia PyTorch-ONNX: OK


In [ ]:
# CPU latency per 224 tile (ORT, batch 1), V3 vs V2 on the same Colab CPU.
# Absolute numbers differ from the dev box (V2: 48 ms there); compare the ratio.
def bench(path, n=50):
    s = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    x = np.random.randn(1, 3, CFG['img_size'], CFG['img_size']).astype(np.float32)
    for _ in range(5):
        s.run(None, {'image': x})
    t0 = time.perf_counter()
    for _ in range(n):
        s.run(None, {'image': x})
    return (time.perf_counter() - t0) / n * 1000


In [ ]:
# Run Benchmark cechk.
lat = {'V3 ' + CFG['encoder']: bench(onnx_path)}
if baseline is not None:
    base_onnx = LOCAL_ROOT / 'baseline_resnet34.onnx'
    baseline.eval().cpu()
    torch.onnx.export(baseline, dummy, str(base_onnx), export_params=True, opset_version=17,
                      do_constant_folding=True, input_names=['image'], output_names=['logits'],
                      dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}}, dynamo=False)
    lat['V2 resnet34'] = bench(base_onnx)

In [ ]:
#  Excecute  torch check to onnx pair
for name, ms in lat.items():
    print(f'{name:<18} {ms:6.1f} ms / tile')
if len(lat) == 2:
    v3, v2 = lat.values()
    print(f'Speedup: {v2 / v3:.2f}x')

/tmp/ipykernel_1191/3420730178.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(baseline, dummy, str(base_onnx), export_params=True, opset_version=17,


V3 mobilenet_v2      59.3 ms / tile
V2 resnet34         122.6 ms / tile
Speedup: 2.07x


## Save deploy artifacts

In [ ]:
k = f"{CFG['deploy_threshold']:.1f}"
test_v3 = results['V3 ' + CFG['encoder']]

# Same schema as ml/configs/model_config.json; ModelConfig.load() reads it as-is.
# No clahe_* keys: CLAHE was never part of training and the backend keeps it off.
deploy_cfg = {
    'model_name':      f"GeoVisionAI {CFG['encoder']}-UNet",
    'model_version':   '2.0.0',
    'architecture':    f"{CFG['encoder']}-UNet",
    'encoder':         CFG['encoder'],
    'encoder_weights': CFG['encoder_weights'],
    'img_size':        CFG['img_size'],
    'in_channels':     CFG['in_channels'],
    'num_classes':     CFG['num_classes'],
    'threshold':       CFG['deploy_threshold'],
    'mean':            CFG['mean'],
    'std':             CFG['std'],
    'onnx':            {'file': ONNX_NAME, 'opset': 17, 'input': 'image', 'output': 'logits'},
    'params':          count_params(model),
    'training': {
        'dataset':          'GeoCrack',
        'best_epoch':       best['epoch'],
        f'val_iou@{k}':     best['val_metrics'][f'iou@{k}'],
        f'test_iou@{k}':    test_v3[f'iou@{k}'],
        f'test_f1@{k}':     test_v3[f'f1@{k}'],
        'test_img_iou@0.5': test_v3['img_iou@0.5'],
        'test_img_f1@0.5':  test_v3['img_f1@0.5'],
        'test_pixel_acc':   test_v3[f'px_acc@{k}'],
    },
    'preprocessing': {
        'sahi_slice_size':    CFG['img_size'],
        'sahi_overlap_ratio': 0.2,
    },
}
cfg_path = DRIVE_OUT / f"model_config_{CFG['encoder']}.json"
cfg_path.write_text(json.dumps(deploy_cfg, indent=2))

table.round(4).to_csv(DRIVE_OUT / 'test_comparison.csv')
for f in ['training_curves.png', 'test_samples.png', 'training_history.json', 'best_iou.pt']:
    if (CHECKPOINTS / f).exists():
        shutil.copy2(CHECKPOINTS / f, DRIVE_OUT / f)

print(json.dumps(deploy_cfg, indent=2))
print('\nArtifacts in', DRIVE_OUT)
for p in sorted(DRIVE_OUT.iterdir()):
    if p.is_file():
        print(f'  {p.name:<36} {p.stat().st_size / 1e6:7.1f} MB')

{
  "model_name": "GeoVisionAI mobilenet_v2-UNet",
  "model_version": "2.0.0",
  "architecture": "mobilenet_v2-UNet",
  "encoder": "mobilenet_v2",
  "encoder_weights": "imagenet",
  "img_size": 224,
  "in_channels": 3,
  "num_classes": 1,
  "threshold": 0.3,
  "mean": [
    0.485,
    0.456,
    0.406
  ],
  "std": [
    0.229,
    0.224,
    0.225
  ],
  "onnx": {
    "file": "geo_vision_mobilenet_v2.onnx",
    "opset": 17,
    "input": "image",
    "output": "logits"
  },
  "params": 6628945,
  "training": {
    "dataset": "GeoCrack",
    "best_epoch": 49,
    "val_iou@0.3": 0.6348188668172677,
    "test_iou@0.3": 0.627866845140956,
    "test_f1@0.3": 0.7713982836066539,
    "test_img_iou@0.5": 0.6078850757116558,
    "test_img_f1@0.5": 0.7419410855899303,
    "test_pixel_acc": 0.9760903360728528
  },
  "preprocessing": {
    "sahi_slice_size": 224,
    "sahi_overlap_ratio": 0.2
  }
}

Artifacts in /content/drive/MyDrive/geocrackai/model_output_v3_mnv2
  best_iou.pt                  

## Deploy to the backend

1. Download from `MyDrive/geocrackai/model_output_v3_mnv2/`:
   - `geo_vision_mobilenet_v2.onnx` → `ml/model/`
   - `model_config_mobilenet_v2.json` → `ml/configs/`
2. In `.env`:
   ```
   MODEL_PATH=ml/model/geo_vision_mobilenet_v2.onnx
   MODEL_CONFIG_PATH=ml/configs/model_config_mobilenet_v2.json
   ```
3. `pytest`, then eyeball overlays on `testImages/` (especially `rocky_crack1.jpg`) against the ResNet34 output
   before switching for good. Keep `geo_vision.onnx` until V3 is confirmed.